# RAG with LangGraph

A simple Retrieval-Augmented Generation pipeline over `islr.pdf` ("An Introduction to Statistical Learning"), built as a 2-node LangGraph:

`START -> retrieve -> generate -> END`

- **retrieve** — embeds the question and pulls the most relevant chunks from a Chroma vector store
- **generate** — asks the Groq LLM to answer using only the retrieved chunks as context

Requires a `GROQ_API_KEY` in a `.env` file (or exported in your shell).


In [1]:
%pip install -q langchain langchain-community langchain-groq langchain-huggingface langchain-chroma pypdf sentence-transformers langgraph python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from typing import TypedDict

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

load_dotenv()

/var/folders/qd/4ypk8rp14qq4q3vbdc7ltg340000gn/T/ipykernel_41590/509646086.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/apekshagangurde/Desktop/Langchain models/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 1. Load and split the PDF


In [3]:
PDF_PATH = "islr.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"Loaded {len(pages)} pages from {PDF_PATH}")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")

Loaded 441 pages from islr.pdf
Split into 1308 chunks


## 2. Embed chunks and build a vector store

Uses a local `sentence-transformers` embedding model (no API key needed). The Chroma index is
persisted to `./chroma_db` so it doesn't have to be rebuilt from scratch on every run.


In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma(
    collection_name="islr",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

if not vector_store.get()["ids"]:
    vector_store.add_documents(chunks)
    print(f"Indexed {len(chunks)} chunks into Chroma")
else:
    print("Chroma collection already populated, skipping re-indexing")

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8328.61it/s]


Indexed 1308 chunks into Chroma


## 3. LangGraph: retrieve -> generate


In [5]:
llm = ChatGroq(model="llama-3.3-70b-versatile")


class RAGState(TypedDict):
    """Graph state — the user's question, the retrieved chunks, and the final answer."""

    question: str
    context: list[Document]
    answer: str


def retrieve(state: RAGState) -> RAGState:
    """Fetch the most relevant PDF chunks for the question from the vector store.

    Args:
        state: The current graph state containing the question.
    """
    docs = retriever.invoke(state["question"])
    return {"context": docs}


def generate(state: RAGState) -> RAGState:
    """Answer the question using only the retrieved chunks as context.

    Args:
        state: The current graph state containing the question and retrieved context.
    """
    context_text = "\n\n".join(doc.page_content for doc in state["context"])
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer isn't in the context, say you don't know.\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {state['question']}"
    )
    response = llm.invoke(prompt)
    return {"answer": response.content}


graph = StateGraph(RAGState)

graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

rag_workflow = graph.compile()

## 4. Test it


In [6]:
result = rag_workflow.invoke({"question": "What is linear regression used for?"})

print("Answer:\n", result["answer"])
print("\nSources:")
for doc in result["context"]:
    print(f"- page {doc.metadata.get('page')}")

Answer:
 Linear regression is used for predicting quantitative values, such as an individual's salary.

Sources:
- page 73
- page 77
- page 20
- page 106
